# Stage 3 — Free-boundary equilibrium

Loads `eq_fixed.h5`, `coilset.h5`, `eq_free.h5`, and `coilset_free.h5` and plots
profile comparisons, LCFS deformation, residual B·n, Boozer spectrum, and force
balance.

Stage 3 co-optimizes **both** the plasma boundary and the coil shapes, so
`eq_free` is self-consistent with `coilset_free` (not the stage-2 `coilset`).
Residual-field and 3-D plots therefore use `coilset_free`.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning, message=".*pynvml.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Unequal number of field periods.*")
os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import matplotlib.pyplot as plt
import plotly.io as pio
from IPython.display import HTML

pio.renderers.render_on_display = False

def show3d(fig):
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

from desc.equilibrium import Equilibrium
from desc.coils import CoilSet
from desc.grid import LinearGrid
from desc.objectives import QuadraticFlux
from desc.plotting import (
    plot_1d,
    plot_section,
    plot_3d,
    plot_coils,
    plot_boozer_surface,
    plot_comparison,
)

HERE = Path(".")

In [ ]:
eq_fixed = Equilibrium.load(str(HERE / "eq_fixed.h5"))
coilset  = CoilSet.load(str(HERE / "coilset.h5"))        # stage-2 coils (fixed-boundary fit)
eq_free  = Equilibrium.load(str(HERE / "eq_free.h5"))
coilset_free = CoilSet.load(str(HERE / "coilset_free.h5"))  # stage-3 co-optimized coils
print(f"Fixed boundary : NFP={eq_fixed.NFP}, L={eq_fixed.L}, M={eq_fixed.M}, N={eq_fixed.N}")
print(f"Coilset (stg2) : {len(coilset.coils)} coils")
print(f"Free boundary  : NFP={eq_free.NFP},  L={eq_free.L}, M={eq_free.M}, N={eq_free.N}")
print(f"Coilset (stg3) : {len(coilset_free.coils)} coils (co-optimized; self-consistent with eq_free)")

## 1-D profile comparison: fixed vs free

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for eq, label, color in [
    (eq_fixed, "fixed boundary", "steelblue"),
    (eq_free,  "free boundary",  "tomato"),
]:
    plot_1d(eq, "iota",     ax=axes[0], color=color, label=label)
    plot_1d(eq, "pressure", ax=axes[1], color=color, label=label)
    plot_1d(eq, "current",  ax=axes[2], color=color, label=label)

axes[0].set_title(r"$\iota(\rho)$"); axes[0].legend()
axes[1].set_title(r"$p(\rho)$ [Pa]")
axes[2].set_title(r"$I(\rho)$ [A]")
fig.suptitle("Stage 3 — profile comparison: fixed vs free boundary", fontsize=13)
plt.tight_layout()
plt.show()

## 2-D cross-section comparison

In [ ]:
N_phi = 4
phi_vals = np.linspace(0, 2 * np.pi / eq_fixed.NFP, N_phi, endpoint=False)

plot_comparison(
    [eq_fixed, eq_free],
    phi=phi_vals,
    color=["steelblue", "tomato"],
    labels=["fixed boundary", "free boundary"],
    rho=5,
    theta=0,
)
plt.suptitle("Stage 3 — LCFS comparison: fixed vs free boundary", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_section(eq_fixed, "|B|", ax=axes[0], log=False)
axes[0].set_title("Fixed boundary — |B| [T]")

plot_section(eq_free, "|B|", ax=axes[1], log=False)
axes[1].set_title("Free boundary — |B| [T]")

fig.suptitle("Stage 3 — |B| cross-section comparison", fontsize=13)
plt.tight_layout()
plt.show()

## 3-D — free-boundary LCFS + coils

In [ ]:
fig = plot_3d(eq_free, "|B|", alpha=0.6)
plot_coils(coilset_free, fig=fig)
fig.update_layout(title="Stage 3 — free-boundary LCFS coloured by |B| + co-optimized coils")
show3d(fig)

## Residual B·n on the free-boundary LCFS

In [ ]:
fig = plot_3d(
    eq_free,
    "B*n",
    field=coilset_free,
    field_grid=LinearGrid(N=50, NFP=eq_free.NFP),
)
plot_coils(coilset_free, fig=fig)
fig.update_layout(title="Stage 3 — residual B·n on free-boundary LCFS (co-optimized coils) [T]")
show3d(fig)

In [ ]:
# Residual B·n, each equilibrium against the coilset it was fit/co-optimized with.
# vacuum=False includes the plasma virtual-casing contribution (correct at finite beta,
# matching the BoundaryError physics used in stage 3).
obj_fixed = QuadraticFlux(eq=eq_fixed, field=coilset, vacuum=False)
obj_fixed.build(verbose=0)
Bn_rms_fixed = float(np.sqrt(obj_fixed.compute_scalar(*obj_fixed.xs(eq_fixed, coilset))))

obj_free = QuadraticFlux(eq=eq_free, field=coilset_free, vacuum=False)
obj_free.build(verbose=0)
Bn_rms_free = float(np.sqrt(obj_free.compute_scalar(*obj_free.xs(eq_free, coilset_free))))

print(f"B·n RMS — fixed eq vs stage-2 coils : {Bn_rms_fixed:.4e} T")
print(f"B·n RMS — free  eq vs stage-3 coils : {Bn_rms_free:.4e} T")
print(f"Reduction factor                    : {Bn_rms_fixed / Bn_rms_free:.2f}×")

## Boozer spectrum

> **Note:** `SHAPE_FACTOR=0.25` attenuates the boundary shaping that made
> precise_QA quasi-axisymmetric, and quasi-symmetry was never an optimization
> target in this pipeline — so a degraded QA spectrum below (non-`B(0,0)`/`B(1,0)`
> modes well above the precise_QA level) is **expected, not a bug**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_boozer_surface(eq_fixed, ax=axes[0])
axes[0].set_title("Fixed boundary — Boozer |B| spectrum")

plot_boozer_surface(eq_free, ax=axes[1])
axes[1].set_title("Free boundary — Boozer |B| spectrum")

fig.suptitle("Stage 3 — Boozer spectrum comparison", fontsize=13)
plt.tight_layout()
plt.show()

## Force balance on the free-boundary equilibrium

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_section(eq_free, "|F|_normalized", ax=axes[0], log=True)
axes[0].set_title("Free boundary — normalised |F| (log)")

plot_1d(eq_free, "|F|", ax=axes[1], color="purple")
axes[1].set_title(r"Free boundary — $|\mathbf{F}|(\rho)$ [N/m³]")

fig.suptitle("Stage 3 — force balance residual", fontsize=13)
plt.tight_layout()
plt.show()

## Verification — recompute errors on finer grids

The `<1%` numbers reported by the stage-3 optimizer are maxima **over the
objective's own evaluation grid**. A denser grid can reveal a higher pointwise
maximum between the optimizer's nodes, so re-evaluate both targets at ~2-3×
resolution before declaring victory. Read the `Maximum ... (normalized)` rows:
both should stay below `1e-2`.

(`QuadraticFlux` with `vacuum=False` is a close proxy for BoundaryError's B·n
row — at a converged solution the sheet current is negligible and they agree.)

In [ ]:
# --- B·n on a finer boundary grid (optimizer used M_grid=16, N_grid=12) ---
from desc.objectives import ForceBalance, ObjectiveFunction

fine_bn = ObjectiveFunction((QuadraticFlux(
    eq=eq_free, field=coilset_free, vacuum=False,
    eval_grid=LinearGrid(M=40, N=40, NFP=eq_free.NFP, sym=False),
),))
fine_bn.build(verbose=0)
fine_bn.print_value(fine_bn.x(eq_free, coilset_free))

In [ ]:
# --- Force balance on a finer volume grid ---
fine_force = ObjectiveFunction((ForceBalance(
    eq=eq_free,
    grid=LinearGrid(L=32, M=32, N=24, NFP=eq_free.NFP, sym=eq_free.sym, axis=False),
),))
fine_force.build(verbose=0)
fine_force.print_value(fine_force.x(eq_free))